# N024 · 数组与字符串的结构变换

**目标：** 利用位置关系完成旋转、编码和就地变换。

**先修：** N017, N018。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 反转分解；循环置换；游程编码；长度与空间契约。

**配套讲解来源：** [同名逐章意见](../../comment/024_array_string_transformations.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个"就地改造序列"的问题，共同要求是不许开新数组（只许常数个额外变量），答案直接写回原数组。

**问题一（数组右旋，`rotate_array`）**：给你一个数组和整数 k，要求把数组整体向右挪 k 个位置——最后 k 个元素搬到最前面，其余依次后移——而且只能原地交换。举个具体到数字的例子：输入 `[1,2,3,4,5]`、k=2 → 输出还是这个数组但内容变成 `[4,5,1,2,3]` → 为什么？向右转 2 格意味着最后两个数 4、5 挪到开头，原来的前三个 1、2、3 顺次跟在后面。麻烦在于 k 可以比数组长度还大（比如 k=7 转一圈半等于只转 2 格），数组还可能是空的，这些都要先处理好。

**问题二（字符压缩，`compress`）**：给你一个字符数组，把"连续重复的段"改写成"字符 + 重复次数"的形式，写回原数组，返回有效长度。例如输入 `['a','a','b']` → 返回 3，且数组前 3 格变成 `['a','2','b']` → 为什么？两个连续的 a 压成 'a','2'，单独一个 b 保持 'b'（重复 1 次不写 "1"）。又如 12 个连续的 a 压成 'a','1','2' 三格。注意返回的是"有效长度"——数组后面的旧字符不许当答案看，这是 “思路推导”部分 特意强调的契约。

两个问题共用一个灵魂：**利用位置关系，用极少的额外空间完成结构性搬运**。

## 2. 基础知识：先读懂这些词

- **右旋（rotate right）**：所有元素下标加 k 再对长度取模的搬运。可以想象数组首尾相接围成一圈，大家顺时针走 k 步。所以转 n 格等于没转，k 大于 n 时只需要 k 除以 n 的余数。
- **取模（`k%=n`）**：k 对 n 取余并把结果存回 k。"转 k 格"和"转 k mod n 格"效果一样，先取模可以把任意大的 k 归到 0 到 n−1 的安全范围，避免做无用功。
- **反转分解（三次反转法）**：右旋 k 格的著名技巧——把数组切成前段 A（长 n−k）和后段 B（长 k），目标是从 AB 变成 BA。做法是：先反转整个数组得到 B 的倒序接 A 的倒序，再把这两段分别反转回来，恰好得到 BA。反转本身只需首尾双指针交换，原地完成。
- **循环置换（cyclic replacement）**：另一类原地旋转思路，沿"下标 +k 取模"的环把元素一个个送到终点。本章实现选的是三次反转法，它比环状置换更好写也更不容易绕晕。
- **双指针读写（read/write pointers）**：压缩用的技术——read 指针负责读原内容，write 指针负责写新内容，两者同步前进。因为压缩结果不会比原文长，write 永远追不上 read，所以"边读边原地覆盖"是安全的。
- **游程编码（run-length encoding，RLE）**：把"连续相同的字符段"（一个"游程"）编码为"字符 + 出现次数"的表示法。这是最古老的数据压缩思想之一：`aaabbb` → `a3b3`。
- **有效长度契约（length contract）**：compress 不修改数组的物理长度（Python 列表也不方便从中间截短），只保证前 k 格是答案，k 由返回值告知。调用方要用 `chars[:k]` 取答案，第 k 格之后是残留的旧数据，不许读。
- **就地变换（in-place）**：只允许常数个额外变量、直接在原数组上修改的算法风格。本章两个函数都是 O(1) 额外空间的就地算法。

## 3. 思路推导：从小例子开始

**`rotate_array` 的思路：**

- **Step 1：挡住空数组。** 长度 n 为 0 时任何旋转都没有意义，而且后面 `k%=n` 会除零报错，所以先 `return` 撤退。
- **Step 2：k 对 n 取模。** 把 k 归到 0..n−1。k=n 时取模得 0，后面三次反转会各做一遍但拼回原样，结果不变。
- **Step 3：写一个反转小工具。** reverse(left,right) 用两个下标从区间两端向中间走，每走一步交换一对元素，相遇即停。它只做原地交换，不占新空间。
- **Step 4：三次反转完成右旋。** 依次反转整个区间 [0,n−1]、前 k 格 [0,k−1]、其余的 [k,n−1]。

**手算演示：`[1,2,3,4,5]`、k=2 三步走（结果对应 “运行示例”部分 表格）**

| 步骤 | 操作 | 数组状态 |
|---|---|---|
| 初始 | — | [1,2,3,4,5]（A=1,2,3；B=4,5） |
| 反转全部 | reverse(0,4) | [5,4,3,2,1]（B 倒序=5,4 接 A 倒序=3,2,1） |
| 反转前 k 格 | reverse(0,1) | [4,5,3,2,1] |
| 反转其余 | reverse(2,4) | [4,5,1,2,3] |

最后的状态正是右旋 2 格的结果。“运行示例”部分 的表格用同一个数组把 k=0..5 全试了一遍：

| 右旋 k | 数组 |
|---|---|
| 0 | [1,2,3,4,5] |
| 1 | [5,1,2,3,4] |
| 2 | [4,5,1,2,3] |
| 3 | [3,4,5,1,2] |
| 4 | [2,3,4,5,1] |
| 5 | [1,2,3,4,5] |

两个规律从表里一眼可见：k 每加 1，就有更多元素从尾部绕回头部；k=5 与 k=0 结果相同（转一整圈等于没转），这正是取模的物理含义。你可以挑 k=3 自己按三步推一遍验证：先反转全部得 [5,4,3,2,1]，再反转前 3 格得 [3,4,5,2,1]，最后反转后 2 格得 [3,4,5,1,2]。

**`compress` 的思路：**

- **Step 1：read、write 双指针从 0 出发。** read 盯着"下一段从哪开始"，write 盯着"答案写到哪了"。
- **Step 2：探测一段游程。** 从 read 出发把 end 向右推，只要当前字符和段首相同就继续，停时 [read,end) 就是完整的一段相同字符，段长是 end−read。
- **Step 3：写出压缩结果。** 先写段首字符；若段长大于 1，再把段长的十进制字符串逐个字符写进去（比如 12 写成 '1'、'2' 两格）。
- **Step 4：read 跳到段尾，循环。** 直到 read 走出数组，返回 write（有效长度）。

**手算演示：`compress(['a','a','b'])` 一段一段走**

| 轮 | read | 探测到的段 | 写入内容 | write 变化 | 数组前几格 |
|---|---|---|---|---|---|
| 1 | 0 | [0,2)：'a','a'，长 2 | 'a' 然后 '2' | 0→2 | a,2,b |
| 2 | 2 | [2,3)：'b'，长 1 | 只写 'b'（不写 1） | 2→3 | a,2,b |

返回 write=3，前 3 格恰是 'a','2','b'；原数组“分段实现”部分之后没有内容，但如果输入更长，后面残留的旧字符一律不算数。再演示多位数：12 个 'a' 一段，长 12，`str(12)` 是两个字符的 '12'，所以写出 'a','1','2' 共 3 格，返回 3——这对应测试里的 `['a']*12` 断言。

## 4. 核心代码：rotate_array

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def rotate_array(nums, k):
    n = len(nums)
    if not n:
        return
    k %= n

    def reverse(left, right):
        while left < right:
            nums[left], nums[right] = (nums[right], nums[left])
            left += 1
            right -= 1
    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)
```

### 逐段读懂代码

### 函数 1：`rotate_array`（原地右旋）

先贴 notebook “分段实现”部分 里的真实代码：

```python
def rotate_array(nums, k):
    n = len(nums)
    if not n:
        return
    k %= n

    def reverse(left, right):
        while left < right:
            nums[left], nums[right] = (nums[right], nums[left])
            left += 1
            right -= 1
    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)
```

- **第 2 行 `n=len(nums)`**：缓存长度，后面取模和三次反转都要用。
- **第 3 行 `if not n: return`**：空数组闸门。空列表 `not n` 为 True，直接返回。这行同时防止第 4 行 `k%=n` 对 0 取模抛 ZeroDivisionError——这是 “思路推导”部分 特意点名的"空数组必须先处理"。
- **第 4 行 `k%=n`**：把 k 归约到 0..n−1。测试里 `rotate_array(a,4)`（n=3）实际转 4%3=1 格，`rotate_array(a,3)` 转 0 格（数组不动）。
- **第 5–8 行 reverse 内嵌函数**：`while left<right` 是"两端的下标还没相遇就继续"；`nums[left],nums[right]=nums[right],nums[left]` 是 Python 的多重赋值交换——右边先整体求值成一对旧值，再同时赋给左边，等价于传统的三行 temp 交换但不需要临时变量；`left+=1; right-=1` 两端向中间收拢一格。整个函数只动两个下标，额外空间 O(1)。
- **第 9 行 `reverse(0,n-1); reverse(0,k-1); reverse(k,n-1)`**：三次反转一条线写完——先整条倒过来，再把前 k 格倒回去，把剩下的 n−k 格倒回去。分号连接三条语句，执行顺序从左到右，顺序不能换（先局部后整体得到的是另一种结果）。k 取模后若为 0，第二次 reverse(0,−1) 因 left>right 一次都不执行，第三次同理，数组保持整条反转再反转的原样，安全。

### 函数 2：`compress`（原地游程压缩）

```python
def compress(chars):
    read = write = 0
    while read < len(chars):
        end = read + 1
        while end < len(chars) and chars[end] == chars[read]:
            end += 1
        chars[write] = chars[read]
        write += 1
        if end - read > 1:
            for c in str(end - read):
                chars[write] = c
                write += 1
        read = end
    return write
```

- **第 2 行 `read=write=0`**：两个指针同时从 0 出发（链式赋值，两个名字绑同一个值 0）。
- **第 4 行 `end=read+1`**：end 是"段尾的下一格"，从段首右边一格开始探测。注意 read 处的字符自己是段的一员，不用比较。
- **第 5 行内层 while**：`end<len(chars)` 防越界（短路保证不会取到 chars[end] 的越界值），`chars[end]==chars[read]` 判断"还是同一个字符"。end 一路推进到第一个不同字符（或数组尽头），此时 [read,end) 是一个极大游程，长度 end−read。
- **第 6 行 `chars[write]=chars[read]; write+=1`**：写出段首字符。无论段长多少，这个字符都要写一次。
- **第 7–8 行 `if end-read>1: for c in str(end-read): chars[write]=c; write+=1`**：段长大于 1 才写计数。`str(end-read)` 把整数变成十进制字符串（12 → '12'），for 逐字符取出，一位数字占一格。段长恰好 1 时整个 if 跳过——这就是"单个字符后面不写 1"的规则出处。
- **第 9 行 `read=end`**：read 直接跳到段尾，开始下一段。中间那些重复字符不需要逐个访问。
- **第 10 行 `return write`**：write 走到哪，答案就到哪长，返回有效长度，履行"长度契约"。
- **为什么 write 不会踩到没读的数据？** 一段长 m≥2 的游程压缩后占"1 个字符 + 位数"格，而位数最多等于 m−1（比如 m=2 占 2 格、m=10 占 3 格、m=12 占 3 格），所以每段写出的格数不超过这段占的格数，write 始终不越过 read，覆盖是安全的——这就是 “正确性与复杂度”部分 里那句"压缩每个已读字符段至少能容纳它的压缩表示"的意思。

## 5. 分段实现：按顺序运行

**本章接口：** `rotate_array(nums, k)`、`compress(chars)`

### rotate_array

In [1]:
def rotate_array(nums, k):
    n = len(nums)
    if not n:
        return
    k %= n

    def reverse(left, right):
        while left < right:
            nums[left], nums[right] = (nums[right], nums[left])
            left += 1
            right -= 1
    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)

### compress

In [2]:
def compress(chars):
    read = write = 0
    while read < len(chars):
        end = read + 1
        while end < len(chars) and chars[end] == chars[read]:
            end += 1
        chars[write] = chars[read]
        write += 1
        if end - read > 1:
            for c in str(end - read):
                chars[write] = c
                write += 1
        read = end
    return write

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,4,5]; rows=[]
for k in range(6):
    b=a[:]; rotate_array(b,k); rows.append((k,b))
show_table(['右旋k','数组'],rows)

右旋k,数组
0,"[1, 2, 3, 4, 5]"
1,"[5, 1, 2, 3, 4]"
2,"[4, 5, 1, 2, 3]"
3,"[3, 4, 5, 1, 2]"
4,"[2, 3, 4, 5, 1]"
5,"[1, 2, 3, 4, 5]"


## 7. 正确性、适用条件与复杂度

把原数组拆为AB，其中B长度为k；反转全部得到BᴿAᴿ，再分别反转得到BA。压缩每个已读字符段至少能容纳它的压缩表示，因此不会写到未读取区域。

**代价：** 均为 O(n) 时间、O(1) 指针空间；计数字符串长度 O(log n)。

### 展开理解

**rotate_array 为什么是对的？** 把数组记成 AB 两段，A 是前 n−k 个、B 是后 k 个，右旋 k 格要的结果正是 BA。三次反转的每一步都可以在纸上核对：第一步反转整个数组，AB 变成"B 整体倒序接 A 整体倒序"；第二步只反转前 k 格，把 B 的倒序翻回正序，此时开头 k 格已经是正确的 B；第三步反转剩下的格子，把 A 的倒序也翻回正序。三步之后数组恰好是 BA，没有任何一步产生多余移动。前置处理也没漏：空数组被提前拦下，k 大于 n 时取模后仍然落在"0 到 n−1 的真实旋转量"范围内，转 n 格等于没转这一点与取模的数学性质吻合。

**compress 为什么是对的？** 外层每轮处理一个极大游程：内层 while 的停止条件保证游程不能再延长，所以任何两个相邻游程的字符必定不同，压缩不漏不串。写出规则与题目契约一致：字符总写一次，计数只在大于 1 时写、且按十进制逐位写。返回值 write 恰好等于写出的格数，所以 chars[:write] 就是完整压缩结果。再配上第四节末尾说的"每段写出的不超过这段占的"，写指针永远毁不掉还没读的内容，整个过程自洽。

**复杂度是多少？** 两个函数都是 O(n) 时间。rotate_array 里每个元素最多被交换两次（整条反转一次、局部反转再一次），n=10⁵ 时大约二十万次交换，一瞬间完成。compress 里 read 和 end 都只前进不后退，总共各走一遍数组，也是线性。额外空间都是 O(1)：rotate 只用 n、k 和 reverse 的两个下标；compress 只用 read、write、end 三个下标。“正确性与复杂度”部分 还提到一个细节：计数字符串 `str(end-read)` 的长度是 O(log n)——一个段最长 n，十进制位数大约是 log₁₀n，n=10⁵ 时计数字符最多 5 个字符，这份临时字符串是压缩函数里唯一非常数空间的瞬间开销，通常也忽略不计。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `a=[1,2,3]; rotate_array(a,4); assert a==[3,1,2]`：测"k 超过长度"。4%3=1，实际右旋 1 格，[1,2,3] 变 [3,1,2]，专门验证取模那行生效。
- `rotate_array(a,3); assert a==[3,1,2]`：测"k 等于长度"。3%3=0，转一整圈回到原位，数组应保持 [3,1,2] 不动，验证零旋转不会误伤数组。
- `a=[]; rotate_array(a,8); assert a==[]`：空数组边界。k=8 看着吓人，但 n=0 直接撤退，既不报错也不改内容，验证那道空数组闸门。
- `a=list('aab'); k=compress(a); assert k==3 and a[:k]==list('a2b')`：压缩正常值。一段长 2 的 a 和一段长 1 的 b，测"长 2 写计数、长 1 不写"的分界，同时用 `a[:k]` 按有效长度取结果，示范长度契约的正确用法。
- `a=['a']*12; k=compress(a); assert a[:k]==list('a12')`：多位计数。12 个 a 压成 'a','1','2'，返回 3，专测"计数字符串逐位写入"那两行。
- **双重循环交叉验证**：`for n in range(1,7): for k in range(12)`——对 1 到 6 的每种长度、0 到 11 的每种 k，先算参考答案 ref（`a[-(k%n):]+a[:-(k%n)] if k%n else a[:]`，即"后 k%n 个接到前面"的切片写法，k%n 为 0 时取原样拷贝），再让 rotate_array 产出并断言相等。这一组是用"另写一个朴素参照实现"来批量对拍，覆盖了包括 k≥n 的 72 种组合，是全组断言里含金量最高的部分。

全部通过后打印 `N024: 所有本章断言通过`。

In [4]:
a = [1, 2, 3]

rotate_array(a, 4)

assert a == [3, 1, 2]

rotate_array(a, 3)

assert a == [3, 1, 2]

a = []

rotate_array(a, 8)

assert a == []

a = list('aab')

k = compress(a)

assert k == 3 and a[:k] == list('a2b')

a = ['a'] * 12

k = compress(a)

assert a[:k] == list('a12')

for n in range(1, 7):
    for k in range(12):
        a = list(range(n))
        ref = a[-(k % n):] + a[:-(k % n)] if k % n else a[:]
        rotate_array(a, k)
        assert a == ref

print('N024: 所有本章断言通过')

N024: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较额外数组与原地旋转。

**练习 2：** 扩展多位次数编码。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（比较额外数组与原地旋转）**：方向是先写"开新数组"的朴素版——`b[(i+k)%n]=a[i]` 一行循环完事——再和本章三次反转版对比。提示：手算用 `[1,2,3,4,5]`、k=2 这个例子，把两种方法各自的数组变化过程列成两张表；然后从三个维度总结——时间都是 O(n)，空间上朴素版要 O(n) 新数组、原地版只有 O(1)，出错面上朴素版几乎不会错、原地版容易在边界（k=0、k=n、空数组）栽跟头。最后用 “自动测试”部分 那个双重循环对拍的方式验证两个版本处处一致。
- **练习 2（扩展多位次数编码）**：方向是确认多位计数已被支持后，继续推广到"次数本身要逐位写入"的更一般情形。提示：先手算 `['a']*123`——压缩结果应是 'a','1','2','3' 四格、返回 4，用 `str(123)` 的逐字符循环就能处理；再考虑边界：段长 1 不写数、段长 10 的两位计数、以及"压缩后总长度比原数组短时如何按有效长度截取答案"。可以用 `compress(list('a'*123))` 之类的调用验证，并自己回答：write 最多可能比 read 落后多少格？

两道练习的验收标准都是"能讲清楚为什么对"，不是"跑通了截图"。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def rotate_array(nums, k):
    n = len(nums)
    if not n:
        return
    k %= n

    def reverse(left, right):
        while left < right:
            nums[left], nums[right] = (nums[right], nums[left])
            left += 1
            right -= 1
    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)

def compress(chars):
    read = write = 0
    while read < len(chars):
        end = read + 1
        while end < len(chars) and chars[end] == chars[read]:
            end += 1
        chars[write] = chars[read]
        write += 1
        if end - read > 1:
            for c in str(end - read):
                chars[write] = c
                write += 1
        read = end
    return write

# 示例与回归测试
a = [1, 2, 3]

rotate_array(a, 4)

assert a == [3, 1, 2]

rotate_array(a, 3)

assert a == [3, 1, 2]

a = []

rotate_array(a, 8)

assert a == []

a = list('aab')

k = compress(a)

assert k == 3 and a[:k] == list('a2b')

a = ['a'] * 12

k = compress(a)

assert a[:k] == list('a12')

for n in range(1, 7):
    for k in range(12):
        a = list(range(n))
        ref = a[-(k % n):] + a[:-(k % n)] if k % n else a[:]
        rotate_array(a, k)
        assert a == ref

print('N024: 所有本章断言通过')

N024: 所有本章断言通过


## 11. 代表题与迁移

- **189. Rotate Array（旋转数组）**：这题就是 `rotate_array` 本尊，练的是三次反转法与"k 先取模、空数组先挡"这两个易错前置处理；原题要求 O(1) 空间，正好逼你放弃开新数组的舒适区。
- **443. String Compression（压缩字符串）**：这题就是 `compress` 本尊，练的是双指针读写 + 按有效长度返回的契约；原题同样要求 O(1) 空间，多位计数（如 12 个 a）是它的经典陷阱。

两题在 notebook 中都标为 canonical（与本章一一对应）。复习时可以自问：为什么 write 永远追不上 read？若允许字符段长 1 也写 '1'，这个性质会被破坏吗（提示：算算 'a' 压成 'a1' 是 1 格变 2 格的情况）？

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。